In [ ]:
import openai
import json
from sklearn.metrics.pairwise import cosine_similarity


In [4]:
def nl_to_ocpq(query):

    prompt = f"""
Translate natural language process queries into OCPQ JSON.

Question: {query}
"""

    response = openai.ChatCompletion.create(
        model="gpt-4",
        messages=[{"role":"user","content":prompt}]
    )

    return json.loads(response["choices"][0]["message"]["content"])

In [ ]:
def retrieve_examples(query, dataset, embeddings, top_k=3):
    query_emb = get_embedding(query)
    
    # Calculate cosine similarity between the query and all dataset questions
    similarities = cosine_similarity([query_emb], embeddings)[0]
    
    # Get the indices of the top-k most similar questions
    top_indices = np.argsort(similarities)[-top_k:][::-1]
    
    examples = []
    for idx in top_indices:
        examples.append({
            "nl": dataset[idx]["nl"],
            "query": dataset[idx]["query"]
        })
    
    return examples

In [ ]:
def nl_to_ocpq_rag(query, dataset, embeddings):
    # Retrieve relevant examples
    examples = retrieve_examples(query, dataset, embeddings)
    
    # Construct a prompt with few-shot examples
    example_prompts = []
    for ex in examples:
        example_prompts.append(
            f"Question: {ex['nl']}\nJSON: {json.dumps(ex['query'])}"
        )
    few_shot_string = "\n\n".join(example_prompts)
    
    prompt = f"""
Translate natural language process queries into OCPQ JSON based on the examples.

EXAMPLES:
{few_shot_string}

Now translate this query:
Question: {query}
"""

    response = openai.ChatCompletion.create(
        model="gpt-4",
        messages=[{"role":"user","content":prompt}]
    )

    return json.loads(response["choices"][0]["message"]["content"])

# Test the function with a new query
# test_query = "merge pull request after close issue"
# print(json.dumps(nl_to_ocpq_rag(test_query, dataset, embeddings), indent=2))